In [1]:
import pandas as pd
import numpy as np
from datetime import datetime



In [2]:

def get_dataframe(variable_name):
    value = globals().get(variable_name)

    if isinstance(value, pd.DataFrame):
        return value.copy()

    return None


def format_table(dataframe, columns=None, decimals=4):
    if dataframe is None or dataframe.empty:
        return "No results available."

    if columns is not None:
        available = [
            column for column in columns
            if column in dataframe.columns
        ]
        dataframe = dataframe[available]

    return dataframe.round(decimals).to_string(index=False)




In [3]:
ood_results = get_dataframe("performance_results")
ood_comparison = get_dataframe("comparison")
confidence_data = get_dataframe("confidence_results")
confident_errors = get_dataframe("all_confident_errors")

cv_results = get_dataframe("cv_results_df")
cv_variance = get_dataframe("variance_results")
cv_stability = get_dataframe("stability_summary")
class_balance = get_dataframe("balance_df")



In [4]:
ood_section = ""
ood_findings = []

if ood_results is not None and not ood_results.empty:

    ood_section = format_table(ood_results)

    baseline_rows = ood_results[
        ood_results["Dataset"] == "Normal Test Data"
    ]

    if not baseline_rows.empty:
        baseline_accuracy = baseline_rows.iloc[0]["Accuracy"]
        baseline_f1 = baseline_rows.iloc[0]["F1 Score"]

        for _, row in ood_results.iterrows():

            if row["Dataset"] == "Normal Test Data":
                continue

            accuracy_change = row["Accuracy"] - baseline_accuracy
            f1_change = row["F1 Score"] - baseline_f1

            ood_findings.append(
                f"- {row['Dataset']}: Accuracy changed by "
                f"{accuracy_change:+.4f}; F1 changed by "
                f"{f1_change:+.4f} compared with normal test data."
            )

else:
    ood_section = "OOD performance results were not available."



In [5]:
confidence_section = "Confidence results were not available."

if confidence_data is not None and not confidence_data.empty:

    confidence_summary = (
        confidence_data.groupby("Dataset")["Confidence"]
        .agg(["mean", "median", "min", "max", "std"])
        .reset_index()
    )

    confidence_section = format_table(confidence_summary)

if confident_errors is not None:
    confident_error_count = len(confident_errors)

    if not confident_errors.empty:
        confident_error_details = format_table(
            confident_errors.head(10)
        )
    else:
        confident_error_details = (
            "No incorrect predictions with confidence "
            "of 80% or higher were found."
        )
else:
    confident_error_count = None
    confident_error_details = (
        "High-confidence error results were not available."
    )



In [6]:
cv_section = "Cross-validation results were not available."
cv_variance_section = "Variance results were not available."
cv_findings = []

if cv_results is not None and not cv_results.empty:

    metrics = [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]

    available_metrics = [
        metric for metric in metrics
        if metric in cv_results.columns
    ]

    cv_summary = (
        cv_results.groupby(["Method", "Folds"])[available_metrics]
        .agg(["mean", "std", "min", "max"])
        .round(4)
    )

    cv_section = cv_summary.to_string()

    if "F1 Score" in cv_results.columns:

        f1_summary = (
            cv_results.groupby(["Method", "Folds"])["F1 Score"]
            .agg(["mean", "std"])
            .reset_index()
        )

        best_cv = f1_summary.loc[
            f1_summary["mean"].idxmax()
        ]

        cv_findings.append(
            f"- Highest average F1 configuration: "
            f"{best_cv['Method']} with {int(best_cv['Folds'])} folds; "
            f"mean F1 = {best_cv['mean']:.4f}."
        )

        most_stable = f1_summary.loc[
            f1_summary["std"].idxmin()
        ]

        cv_findings.append(
            f"- Lowest F1 standard deviation: "
            f"{most_stable['Method']} with "
            f"{int(most_stable['Folds'])} folds; "
            f"standard deviation = {most_stable['std']:.4f}."
        )

if cv_variance is not None and not cv_variance.empty:
    cv_variance_section = format_table(cv_variance)



In [7]:
if class_balance is not None and not class_balance.empty:
    class_balance_section = format_table(class_balance)
else:
    class_balance_section = "Class balance results were not available."



In [8]:
report = f"""
# TECHNICAL REPORT
# Model Robustness Testing and Generalization Analysis

**Project:** Heart Disease Classification

**Model:** Random Forest Classifier

**Report generated:** {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

---

## 1. Executive Summary

This report evaluates the robustness and generalization behaviour
of a heart disease classification model.

The experiments include:

- Adversarial and feature perturbation testing.
- Synthetic out-of-distribution testing.
- Model confidence analysis.
- High-confidence prediction error analysis.
- Custom K-Fold cross-validation.
- Custom Stratified K-Fold cross-validation.
- Performance variance analysis across different fold counts.

The purpose is to understand whether the model maintains
consistent predictions when input data changes and whether
its performance is stable across different validation samples.

## 2. Experimental Setup

### 2.1 Dataset

The experiments use the heart disease classification dataset.

The target variable is `target`.

The data is divided into input features and the target class.
Duplicate records and rows with missing target values are removed.

### 2.2 Model Configuration

The Random Forest classifier uses the following configuration:

- Number of estimators: 150
- Maximum tree depth: 8
- Class weighting: Balanced
- Random state: 42
- Missing-value handling: Median imputation

### 2.3 Robustness Experiments

The model is examined using three main testing approaches.

**A. Adversarial and perturbation testing**

Input features are modified to examine how sensitive predictions
are to small or deliberately selected changes.

**B. Out-of-distribution testing**

Synthetic high-value, low-value, and mixed edge cases are created
by moving continuous features beyond their observed training ranges.

**C. Cross-validation**

Custom K-Fold and Stratified K-Fold mechanisms are tested using
different fold counts and repeated random assignments.

---

## 3. Out-of-Distribution Performance Results

The following table compares model performance on normal test
samples and synthetic OOD samples.

{ood_section}

### 3.1 Performance Trends

{chr(10).join(ood_findings) if ood_findings else "No OOD comparison findings could be calculated."}

A decrease in accuracy or F1 score indicates sensitivity to the
synthetic feature changes.

However, these measurements must be interpreted cautiously.
The original test labels are reused after modifying the features.
The modified features may no longer represent the original
diagnosis, so OOD accuracy is an exploratory stress-test measure.

---

## 4. Model Confidence Analysis

The model confidence results are summarized below.

{confidence_section}

Confidence measures the highest predicted class probability.

A high confidence value does not guarantee that a prediction
is correct. A model may remain highly confident even when its
input is unfamiliar or outside its training distribution.

### 4.1 High-Confidence Incorrect Predictions

Total high-confidence incorrect predictions:
{confident_error_count if confident_error_count is not None else "Not available"}

{confident_error_details}

These cases deserve attention because they indicate situations
where the model may provide a confident but incorrect output.

---

## 5. Cross-Validation Performance

The model was evaluated using custom K-Fold and custom
Stratified K-Fold splitting.

Different fold counts and repeated random seeds were used.

### 5.1 Performance Summary

{cv_section}

### 5.2 Variance Analysis

{cv_variance_section}

### 5.3 Cross-Validation Findings

{chr(10).join(cv_findings) if cv_findings else "Cross-validation findings could not be calculated."}

A lower standard deviation suggests more consistent performance
across the tested folds and repetitions.

Stratified splitting aims to preserve the overall class
proportions in each validation fold. This is especially useful
when class frequencies are unequal.

---

## 6. Class Representation

The class distribution in the validation folds was examined.

{class_balance_section}

Stratification helps prevent folds from containing substantially
different class proportions.

It does not guarantee equal class counts in every fold, and it
does not balance other characteristics such as age or sex.

---

## 7. Failure Modes and Limitations

The experiments highlight several potential failure modes.

### 7.1 Sensitivity to Feature Changes

Predictions may change when input features are perturbed.
This indicates that the model may rely heavily on particular
features or feature combinations.

### 7.2 Unfamiliar Input Values

Synthetic edge cases may lie outside the feature ranges observed
during training. Tree-based models can still return predictions
for these inputs, even when they are unfamiliar.

### 7.3 Confident Incorrect Predictions

High-confidence errors indicate that prediction confidence
should not be treated as proof of reliability.

### 7.4 Validation Variability

Different fold assignments may produce different performance
scores. High variance can indicate that model performance is
sensitive to the selected training and validation samples.

### 7.5 Synthetic Data Limitations

Artificially generated edge cases may not represent realistic
patients. Extreme feature combinations can also be medically
implausible.

### 7.6 External Generalization

Cross-validation and synthetic testing use the same underlying
dataset. They cannot prove that the model will generalize to
patients from another hospital, region, or time period.

---

## 8. Recommendations for Improving Model Resilience

1. **Use external validation:** Test the model on an independent
   dataset from another source or patient population.

2. **Improve training-data coverage:** Include a wider range of
   representative and clinically valid observations.

3. **Protect feature validity:** Apply medically justified
   constraints to continuous and categorical input values.

4. **Monitor confidence:** Flag uncertain predictions for further
   review instead of automatically trusting every output.

5. **Investigate confident errors:** Examine high-confidence
   incorrect cases to identify recurring feature patterns.

6. **Use stratified validation:** Preserve class proportions when
   comparing model configurations.

7. **Measure performance variation:** Report both average
   performance and standard deviation.

8. **Reassess feature sensitivity:** Identify features whose
   perturbations frequently change model predictions.

9. **Evaluate calibration:** Check whether predicted probabilities
   correspond to observed outcome frequencies.

10. **Monitor model drift:** Re-evaluate performance when the
    distribution of incoming data changes.

---

## 9. Overall Conclusion

The robustness experiments provide a structured assessment
of model sensitivity, confidence, and internal validation stability.

Synthetic OOD testing examines the model's behaviour on
unfamiliar feature values. Perturbation testing examines
prediction sensitivity. Cross-validation measures performance
consistency across different data partitions.

These experiments are useful for identifying weaknesses,
but they do not establish clinical safety or real-world
generalization.

Before any clinical use, the model requires external validation,
clinical review, probability calibration, and ongoing monitoring.

---

## 10. Generated Supporting Files

The following files can be used for further analysis:

- ood_performance_results.csv
- ood_confidence_results.csv
- ood_confident_errors.csv
- ood_generalization_report.csv
- custom_cross_validation_fold_results.csv
- custom_cross_validation_variance.csv
- custom_cross_validation_stability.csv

Only files successfully generated by the corresponding experiment
cells should be treated as available.
"""



In [9]:


with open(
    "model_robustness_technical_report.md",
    "w",
    encoding="utf-8"
) as file:
    file.write(report)

with open(
    "model_robustness_technical_report.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(report)

print("Technical report generated successfully.")
print("File 1: model_robustness_technical_report.md")
print("File 2: model_robustness_technical_report.txt")

print("\nREPORT PREVIEW")
print("=" * 60)
print(report[:5000])


Technical report generated successfully.
File 1: model_robustness_technical_report.md
File 2: model_robustness_technical_report.txt

REPORT PREVIEW

# TECHNICAL REPORT
# Model Robustness Testing and Generalization Analysis

**Project:** Heart Disease Classification

**Model:** Random Forest Classifier

**Report generated:** 2026-11-16 09:47:51

---

## 1. Executive Summary

This report evaluates the robustness and generalization behaviour
of a heart disease classification model.

The experiments include:

- Adversarial and feature perturbation testing.
- Synthetic out-of-distribution testing.
- Model confidence analysis.
- High-confidence prediction error analysis.
- Custom K-Fold cross-validation.
- Custom Stratified K-Fold cross-validation.
- Performance variance analysis across different fold counts.

The purpose is to understand whether the model maintains
consistent predictions when input data changes and whether
its performance is stable across different validation samples.

## 2